03 冻结参数样本外（2020 起）+ Walk-forward

步骤：IS 里选参（邻域平均 Sharpe，回撤不劣于基线）→ 冻结 → 只在 OOS 评估一次。

沿用原有连续历史回测状态和逐年收益拼接口径；测试年份仍为 2016–2026，缺少数据的年份跳过。

在 PyCharm 中选择 Python 内核，点击 **Run All / 运行全部**。最后一格调用 `main()`，无需手动逐个调用函数。


In [1]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """支持从项目根目录、pipeline 目录及项目内子目录启动。"""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file() and (candidate / '07_run_pipeline.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('找不到 pipeline 文件夹。请将 PyCharm 工作目录设为项目根目录，或设置 ETF_PROJECT_ROOT。')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """独立命名空间加载函数；不执行被加载 notebook 的入口单元格。"""
    path = PIPELINE_DIR / filename
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [2]:
def evaluate_frozen_parameters(C, runs):
    best,sc=select(runs,IS_START,IS_END)
    print('IS 选中（冻结）:',label(best),'| IS Sharpe',round(sc[best]['Sharpe'],3))
    rows=[]
    for name,c in [('基线',base),('选中',best)]:
        for lab,a,b in [('IS',IS_START,IS_END),('OOS',OOS_START,None),('OOS剔除2020','2021-01-01',None)]:
            rows.append({'方案':name,'区间':lab,**fmt(metrics(runs[c][0],a=a,b=b))})
    for t in ['SPY','TQQQ']:
        for lab,a in [('OOS',OOS_START),('OOS剔除2020','2021-01-01')]:
            rows.append({'方案':'买入持有 '+t,'区间':lab,**fmt(metrics(C[t].loc[a:],))})
    table = pd.DataFrame(rows)
    display(table)
    return {'best': best, 'scores': sc, 'metrics': table}


In [3]:
def compare_oos(runs, sc, best):
    # 全部组合 vs 基线（OOS）——用字符串作键，避免 None 在 pandas 索引里变成 NaN
    def cmp(a):
        bm=metrics(runs[base][0],a=a)
        oo={label(c):metrics(runs[c][0],a=a) for c in combos if c!=base}
        return bm,oo
    for lab,a in [('OOS 2020+',OOS_START),('OOS 2021+(剔除2020)','2021-01-01')]:
        bm,oo=cmp(a)
        print(lab,'| 基线 Sharpe',round(bm['Sharpe'],3),'| 超过基线的组合数:',sum(m['Sharpe']>bm['Sharpe'] for m in oo.values()),'/',len(oo),
              '| 回撤更浅:',sum(m['MDD']>bm['MDD'] for m in oo.values()),'| CAGR更高:',sum(m['CAGR']>bm['CAGR'] for m in oo.values()))
    bm,oo=cmp(OOS_START)
    isS=pd.Series({label(c):sc[c]['Sharpe'] for c in combos if c!=base}); osS=pd.Series({k:m['Sharpe'] for k,m in oo.items()})
    print('IS 与 OOS 的 Sharpe 排名相关（Spearman）:',round(isS.corr(osS,method='spearman'),2))
    print('选中参数的 OOS 排名:',int(osS.rank(ascending=False)[label(best)]),'/',len(osS))
    return {'is_sharpe': isS, 'oos_sharpe': osS}


In [4]:
def evaluate_walkforward(runs):
    pieces_sel=[];pieces_base=[];log=[]
    for ty in range(2016,2027):
        te_a,te_b=f'{ty}-01-01',f'{ty}-12-31'
        tr_a=f'{ty-4}-01-01' if ty-4>2012 else IS_START; tr_b=f'{ty-1}-12-31'
        if runs[base][0].loc[te_a:te_b].shape[0] < 2:
            print(f'{ty} 年数据不足，跳过该测试年。')
            continue
        b_,_=select(runs,tr_a,tr_b)
        rs=runs[b_][0].loc[te_a:te_b].pct_change().dropna(); rb=runs[base][0].loc[te_a:te_b].pct_change().dropna()
        pieces_sel.append(rs);pieces_base.append(rb)
        log.append({'测试年':ty,'选中参数':label(b_),'选中收益':round((1+rs).prod()-1,3),'基线收益':round((1+rb).prod()-1,3)})
    display(pd.DataFrame(log))
    def from_r(r):
        e=(1+pd.concat(r)).cumprod(); return pd.concat([pd.Series([1.0],index=[e.index[0]-pd.Timedelta(days=1)]),e])
    rows=[]
    for nm,ps in [('WF 选中',pieces_sel),('WF 基线',pieces_base)]:
        rows.append({'方案':nm,'区间':'全部',**fmt(metrics(from_r(ps)))})
        rows.append({'方案':nm,'区间':'剔除2020',**fmt(metrics(from_r([p for p in ps if p.index[0].year!=2020])))})
    table = pd.DataFrame(rows)
    display(table)
    return {'yearly_log': pd.DataFrame(log), 'metrics': table, 'selected_returns': pieces_sel, 'baseline_returns': pieces_base}


In [5]:
def main(csv_path=None, context=None, runs=None):
    """本 notebook 的一键入口；返回表格、净值等结果。"""
    context = prepare_context(csv_path) if context is None else context
    O, H, L, C, tg = context
    runs = run_all(*context) if runs is None else runs
    frozen = evaluate_frozen_parameters(C, runs)
    comparison = compare_oos(runs, frozen['scores'], frozen['best'])
    walkforward = evaluate_walkforward(runs)
    return {'frozen': frozen, 'comparison': comparison, 'walkforward': walkforward}


In [6]:
# 修改数据路径后，运行全部单元格。None 表示自动查找项目 data/etf_daily.csv。
CSV_PATH = None
results = main(csv_path=CSV_PATH)


数据区间: 2011-10-03 ~ 2026-10-07 | 信号分布: {'TQQQ': 2904, 'TECL': 249, 'TECS': 203, 'BSV': 150, 'UVXY': 133, 'SPXL': 17}


IS 选中（冻结）: 固定 止损=0.12 止盈=0.15 | IS Sharpe 0.85


,方案,区间,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,基线,IS,0.446,0.497,0.989,1.490,-0.738,0.604
1,基线,OOS,1.584,0.730,1.663,2.686,-0.508,3.117
2,基线,OOS剔除2020,0.939,0.656,1.335,2.091,-0.508,1.847
3,选中,IS,0.285,0.380,0.850,1.266,-0.621,0.458
4,选中,OOS,0.967,0.526,1.540,2.741,-0.522,1.851
5,选中,OOS剔除2020,0.554,0.440,1.224,1.920,-0.522,1.060
6,买入持有 SPY,OOS,0.154,0.200,0.817,1.158,-0.337,0.456
7,买入持有 SPY,OOS剔除2020,0.154,0.166,0.945,1.378,-0.245,0.627
8,买入持有 TQQQ,OOS,0.353,0.734,0.785,1.116,-0.817,0.433
9,买入持有 TQQQ,OOS剔除2020,0.275,0.665,0.699,1.005,-0.817,0.337


OOS 2020+ | 基线 Sharpe 1.663 | 超过基线的组合数: 3 / 35 | 回撤更浅: 26 | CAGR更高: 1
OOS 2021+(剔除2020) | 基线 Sharpe 1.335 | 超过基线的组合数: 2 / 35 | 回撤更浅: 29 | CAGR更高: 0


IS 与 OOS 的 Sharpe 排名相关（Spearman）: 0.57
选中参数的 OOS 排名: 7 / 35


,测试年,选中参数,选中收益,基线收益
0,2016,追踪 止损=0.06 止盈=0.25,0.138,0.811
1,2017,追踪 止损=None 止盈=0.4,1.335,1.335
2,2018,追踪 止损=None 止盈=0.15,-0.457,-0.539
3,2019,固定 止损=0.12 止盈=0.15,1.214,1.559
4,2020,固定 止损=0.12 止盈=0.15,6.679,13.229
5,2021,固定 止损=0.12 止盈=0.15,0.363,1.676
6,2022,追踪 止损=None 止盈=0.15,-0.023,0.166
7,2023,追踪 止损=None 止盈=0.4,0.843,1.534
8,2024,追踪 止损=None 止盈=0.4,0.652,0.682
9,2025,追踪 止损=None 止盈=0.4,1.305,1.093


,方案,区间,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,WF 选中,全部,0.720,0.557,1.246,2.080,-0.641,1.123
1,WF 选中,剔除2020,0.423,0.514,1.013,1.580,-0.641,0.660
2,WF 基线,全部,1.120,0.662,1.467,2.331,-0.738,1.516
3,WF 基线,剔除2020,0.656,0.606,1.222,1.891,-0.738,0.888
